In [ ]:
from math import *
from build123d import *
from cad_utils import *
from pcb import PCB

set_port(3939)
set_viewer_config(port=3939)

In [ ]:
reset_show()

pcb = PCB.pcb
pcb_bottom_face: Face = pcb.faces().group_by(Axis.Z)[0].first
pcb_outer = pcb_bottom_face.outer_wire()
pcb_size = pcb_outer.bounding_box().size

pcb_usb = PCB.find('USB_TYPE_C_PORT')
pcb_usb_outer = pcb_usb.faces().group_by(Axis.Y)[-1].first.outer_wire()

pcb_switch = PCB.find('S2')
pcb_switch_face = pcb_switch.faces().group_by(Axis.Y)[-1].first

pcb_screen = PCB.find("J1 Screen")
pcb_screen_top_face = pcb_screen.faces().filter_by(Plane.XY).sort_by(Axis.Z)[-2]
pcb_screen_screen_face = pcb_screen.faces().filter_by(Plane.XY).sort_by(Axis.Z)[-1]

pcb_encoder = PCB.find("S1")
pcb_sliders = [PCB.find("R7"), PCB.find("R8")]
pcb_slider_levers = [PCB.find("R7", "Lever"), PCB.find("R8", "Lever")]

show_object(pcb_outer)
# show_object(pcb_usb)
show_object(pcb_usb_outer)
# show_object(pcb_switch)
show_object(pcb_switch_face)
show_object(pcb_screen)
show_object(pcb_screen_top_face)
show_object(pcb_screen_screen_face)
show_object(pcb_slider_levers)


In [ ]:
pcb_lip = 2
wall_thickness = 1.8
wall_offset = 0.5
basement_height = 10.0

bottom_thickness = 1.0
top_thickness = 1.0
switch_cutout_r = 4
lid_interface = wall_thickness
lid_gap = 0.25
cutout_extra = 0.25
screen_cutout_extra = 1

antenna_width = 40
antenna_height = 20
antenna_depth = 0.6
antenna_inset = 0.4

lid_wedge_size = 10
lid_wedge_relief_r = 0.1
lid_wedge_subtract_oversize = lid_wedge_relief_r * 2

main_height = pcb_screen.bounding_box().max.Z + 0.2

screen_standoffs_uv = [(0.25, 0.1), (0.75, 0.1)]
screen_standoffs_size = 3

screen_lid_indent = 0.4

battery_size = (30, 41.5, 5.0)
battery_tolerance = 0.2


In [ ]:
reset_show()


def lid_wedge(mode=Mode.ADD, size=lid_wedge_size):
  """Create a 45 degree wedge. Its "bottom" is centered in x_dir going towards +z, and its "point" is towards +y."""
  w = size
  h = lid_interface
  return Wedge(
    xsize=w,
    ysize=sqrt(h),
    zsize=h,
    xmin=h / 2,
    zmin=h / 2,
    xmax=w - h / 2,
    zmax=h / 2,
    align=(Align.CENTER, Align.MIN, Align.MIN),
    mode=mode,
  )


def pcb_outer_offset_sketch(offset_amount, z=None):
  with BuildSketch(Plane.XY.offset(z) if z else Plane.XY, mode=Mode.PRIVATE) as s:
    offset(pcb_outer.edges(), amount=offset_amount)
    make_face()
  return s.sketch


with BuildPart() as case_bottom:
  extrude(
    pcb_outer_offset_sketch(wall_offset + wall_thickness),
    amount=-bottom_thickness - basement_height,
  )
  extrude(
    pcb_outer_offset_sketch(wall_offset + wall_thickness),
    amount=main_height,
  )
  # cut basement cavity
  extrude(
    pcb_outer_offset_sketch(wall_offset - pcb_lip),
    amount=-basement_height,
    mode=Mode.SUBTRACT,
  )
  # cut main cavity
  extrude(
    pcb_outer_offset_sketch(wall_offset),
    amount=main_height,
    mode=Mode.SUBTRACT,
  )

  # add lid wedges
  inner_y_faces = faces(Select.LAST).filter_by(Plane.XZ)
  wedge_planes = []
  for f in inner_y_faces:
    for u, x in [(0, lid_wedge_size / 2), (0.5, 0), (1, -lid_wedge_size / 2)]:
      l = f.location_at(u, 0)
      p = Plane(l)
      p.origin += Vector(x * p.x_dir.X, 0, 0)
      wedge_planes.append(Plane(p.origin, x_dir=p.x_dir, y_dir=-p.z_dir))
  with Locations(wedge_planes):
    lid_wedge()

  # add usb cutout
  back = faces().sort_by(Axis.Y).last

  with BuildSketch(back) as s:
    l = project(pcb_usb_outer, mode=Mode.PRIVATE)
    make_face(l)
    o = offset(faces(), amount=cutout_extra, side=Side.RIGHT)
  extrude(amount=-wall_thickness, mode=Mode.SUBTRACT)

  # add switch cutout
  with BuildSketch(pcb_switch_face.project_to_shape(back, Axis.Y.direction)) as s:
    Circle(switch_cutout_r + cutout_extra)
  extrude(amount=-wall_thickness, mode=Mode.SUBTRACT)

  # add antenna groove
  top: Face = faces().filter_by(Plane.XY).sort_by(Axis.Z)[-1]
  top_left_edge_inner: Edge = (
    top.edges().filter_by(Plane.YZ).filter_by(GeomType.LINE).sort_by(Axis.X)[1]
  )
  antenna = top_left_edge_inner.end_point() + Vector(
    -antenna_inset - antenna_depth / 2, -antenna_width / 2, 0
  )
  with BuildSketch(Plane(antenna)) as s:
    SlotArc(Edge.make_line((0, -antenna_width / 2), (0, antenna_width / 2)), height=antenna_depth)
  extrude(amount=-antenna_height, mode=Mode.SUBTRACT)
  with BuildSketch(Plane(antenna)) as s:
    Rectangle(wall_thickness, antenna_width / 3, align=(Align.MIN, Align.CENTER))
  extrude(amount=-antenna_height / 3 * 2, mode=Mode.SUBTRACT)

  # pcb standoff wedges
  inner_back: Face = faces().filter_by(Plane.XZ).sort_by(Axis.Y)[-2]
  screen_bottom = pcb_screen.faces().filter_by(Plane.XY).sort_by(SortBy.AREA).last
  screen_standoff_planes = [
    Plane(screen_bottom.position_at(u, v), (0, 1, 0), (1, 0, 0)) for u, v in screen_standoffs_uv
  ]
  screen_standoff_distance = inner_back.distance_to(
    screen_bottom.position_at(*screen_standoffs_uv[0])
  )
  with BuildSketch(screen_standoff_planes) as s:
    Triangle(b=screen_standoff_distance, B=45, C=90, align=Align.MAX, rotation=90)
  extrude(amount=-screen_standoffs_size / 2, both=True)
  chamfer(edges(Select.LAST).group_by(SortBy.LENGTH)[-1], screen_standoffs_size / 4)

  # battery ridges
  bottom_inner = faces().filter_by(Plane.XY).sort_by(Axis.Z)[1]
  with BuildSketch(bottom_inner) as s:
    rect = Rectangle(battery_size[0]+wall_thickness+battery_tolerance, battery_size[1]+wall_thickness+battery_tolerance, mode=Mode.PRIVATE)
    for e in [e.trim(1/4, 3/4) for e in rect.edges()]:
      with BuildLine() as l:
        offset(e, wall_thickness/2)
      make_face()
  extrude(amount=battery_size[2], mode=Mode.ADD)

case_bottom = case_bottom.part
case_bottom.label = "case_bottom"
case_bottom.color = Color(0.2, 0.2, 0.2)
show_object(case_bottom)

In [ ]:
reset_show()

with BuildPart() as case_top:
  extrude(
    pcb_outer_offset_sketch(wall_offset + wall_thickness, z=main_height),
    amount=top_thickness,
  )
  lip_full = extrude(
    pcb_outer_offset_sketch(wall_offset - lid_gap, z=main_height),
    amount=-lid_interface,
  )
  extrude(
    pcb_outer_offset_sketch(wall_offset - lid_gap - wall_thickness, z=main_height),
    amount=-lid_interface,
    mode=Mode.SUBTRACT,
  )

  p = project(
      pcb_screen_top_face,
      workplane=Plane(faces().sort_by(SortBy.AREA)[-2]),
      mode=Mode.PRIVATE,
    ).faces().sort_by(Axis.Z).first
  pp = offset(p, amount=screen_cutout_extra, mode=Mode.PRIVATE)
  extrude(pp, amount=-screen_lid_indent, mode=Mode.SUBTRACT)
  

  p = project(
      pcb_screen_screen_face,
      workplane=Plane(faces().sort_by(SortBy.AREA)[-2]),
      mode=Mode.PRIVATE,
    ).faces().sort_by(Axis.Z).first
  pp = offset(p, amount=screen_cutout_extra, mode=Mode.PRIVATE)
  extrude(pp, amount=-wall_thickness, taper=-45, mode=Mode.SUBTRACT)

  outer_y_faces = lip_full.faces().filter_by(Plane.XZ)

  fillet(outer_y_faces.edges().sort_by(Axis.Z).first, wall_thickness/2)

  wedge_planes = []
  for f in outer_y_faces:
    for u, x in [(0, lid_wedge_size / 2), (0.5, 0), (1, -lid_wedge_size / 2)]:
      l = f.location_at(u, 1)
      p = Plane(l)
      p.origin += Vector(x * p.x_dir.X, 0, 0)
      wedge_planes.append(Plane(p.origin, x_dir=p.x_dir, y_dir=p.z_dir))


  with Locations(wedge_planes):
    with Locations((0, wall_thickness, 0)):
      lid_wedge()
    lid_wedge(mode=Mode.SUBTRACT)
   

  top = faces().sort_by(SortBy.AREA).last
  for slider, lever in zip(pcb_sliders, pcb_slider_levers):
    slider_bbox = slider.bounding_box()
    slider_center_plane = Plane(slider_bbox.center(), x_dir=(1, 0, 0), y_dir=(0, 0, 1))
    section: Face = lever.intersect(top)[0]
    c1: Location = section.center_location
    c2 = c1.mirror(slider_center_plane)
    radius = section.bounding_box().size.X + cutout_extra
    slot_location = Plane((c1.position + c2.position) / 2, x_dir=(0, 1, 0), y_dir=(1, 1, 0))
    length = (
      Edge.make_line(c1.position, c2.position).length
      + section.bounding_box().size.Y
      - radius
      + cutout_extra * 2
    )
    with BuildSketch(slot_location) as s:
      SlotCenterToCenter(length, radius)
    extrude(amount=wall_thickness, mode=Mode.SUBTRACT)

  # Encoder cutout - find the circular shaft cross-section at the top face
  encoder_section = pcb_encoder.intersect(top)
  encoder_face = encoder_section.faces().sort_by(SortBy.AREA).last
  encoder_circle = encoder_face.edges().filter_by(GeomType.CIRCLE).first.geom_adaptor().Circle()
  encoder_center = Vector(encoder_circle.Position().Location())
  encoder_radius = encoder_circle.Radius()
  with BuildSketch(Plane(encoder_center)) as s:
    Circle(encoder_radius + cutout_extra)
  extrude(amount=-wall_thickness, mode=Mode.SUBTRACT)

case_top = case_top.part
case_top.label = "case_top"
case_top.color = Color(0.2, 0.2, 0.2)
show_object(case_top)

show_object(pcb_screen)

In [ ]:
export(case_bottom)
export(case_top)

In [ ]:
case = Compound(None, label="case", children=[
  fast_copy(case_bottom),
  fast_copy(case_top),
])